# Fashion-MNIST

**Objetivo:** usar la arquitectura ganadora del notebook de MNIST (1 capa oculta densa) para reconocer ropa (Fashion MNIST) y comparar resultados.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## 1. Carga y exploración: ¿qué datos tenemos?

Fashion MNIST trae 70.000 imágenes de 28x28 píxeles en escala de grises (60.000 para entrenar, 10.000 para test), igual formato que MNIST pero con ropa en vez de dígitos. Cada imagen tiene una etiqueta del 0 al 9 según la prenda.

In [ ]:
# Nombres de las 10 clases de Fashion MNIST (en español)
nombres_clases = ["remera", "pantalón", "suéter", "vestido", "abrigo", "sandalia", "camisa", "zapatilla", "bolso", "bota"]

# Descargamos Fashion MNIST desde Keras (viene incluido)
(X_train_img, y_train), (X_test_img, y_test) = keras.datasets.fashion_mnist.load_data()

# Mostrar cantidad de imágenes y sus dimensiones
print("Train:", X_train_img.shape)
print("Test:", X_test_img.shape)

# Mostramos 10 ejemplos para ver las prendas y sus etiquetas
n_mostrar = 10
plt.figure(figsize=(10, 2))
for i in range(n_mostrar):
    plt.subplot(1, n_mostrar, i + 1)
    plt.imshow(X_train_img[i], cmap="gray")
    plt.title(nombres_clases[y_train[i]])
    plt.axis("off")
plt.show()

## 2. Preprocesamiento: normalizar y aplanar

- **Normalizar:** los píxeles van de 0 a 255. Los dividimos por 255 para dejarlos entre 0 y 1, así la red aprende más rápido (igual que en MNIST).
- **Aplanar:** la red densa no entiende imágenes 2D, así que cada foto de 28x28 la convertimos en un vector de 784 números.

In [ ]:
# Normalizamos a 0-1 y aplanamos de 28x28 a vector de 784
INPUT_DIM = 28 * 28
X_train = (X_train_img.astype("float32") / 255.0).reshape(-1, INPUT_DIM)
X_test = (X_test_img.astype("float32") / 255.0).reshape(-1, INPUT_DIM)

print("Ejemplo train aplanado:", X_train.shape)  # (60000, 784)
print("Valor mínimo y máximo:", X_train.min(), X_train.max())

## 3. Arquitectura: (ganadora de MNIST)


Repetimos el proceso con Fashion MNIST **usando la arquitectura ganadora del experimento anterior**

- **Entrada:** 784 neuronas (una por píxel).
- **Capa oculta:** 128 neuronas con activación ReLU o Sigmoid (eso lo comparamos después).
- **Salida:** 10 neuronas con `softmax` (una probabilidad por cada prenda 0-9).

La mantenemos igual a propósito para que la comparación MNIST vs Fashion sea justa: si cambia el accuracy, es por el dataset (ropa es más difícil que dígitos), no por la red.

In [ ]:
def crear_modelo(activacion, optimizador):
    lr = 0.001
    if optimizador == "adam":
        opt = keras.optimizers.Adam(learning_rate=lr)
    else:
        opt = keras.optimizers.SGD(learning_rate=lr)

    modelo = keras.Sequential([
        keras.layers.Input(shape=(784,),name="entrada"),
        keras.layers.Dense(128, activation=activacion, name="oculta"),
        keras.layers.Dense(10, activation="softmax", name="salida")
    ])

    modelo.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modelo

# Mostramos el resumen ReLU + Adam como ejemplo (ganadora en MNIST)
modelo_demo = crear_modelo(activacion="relu", optimizador="adam")
modelo_demo.summary()